# Step 02 — Introduction to CrewAI

🇬🇧 **English** (this notebook)

A structured look at CrewAI itself — the framework Step 01's standalone `Agent` came from. We start with a plain LLM call, watch it forget the conversation after one turn, then bring in CrewAI's `Agent`/`Task`/`Crew` to solve that properly.

This notebook covers only what you'll actually use for the rest of this course. CrewAI itself is much bigger — Flows, hierarchical processes, the full memory system, ~90 built-in tools, and more. For any of that, the [CrewAI documentation](https://docs.crewai.com) is the reference; the most relevant pages are linked under "Resources for further reading" at the end.

## Learning objective

By the end of this notebook, you will:

- Understand what CrewAI is and when to reach for it over a plain LLM call
- Understand CrewAI's core abstractions — `Agent`, `Task`, `Crew`, `Process` — and how they fit together
- Be able to work with `role`, `goal`, and `backstory` as CrewAI's version of prompts and system messages
- Have used CrewAI's built-in memory (`memory=True`) to give an agent multi-turn recall
- Understand what `tracing=True` gives you over `verbose=True` — a shareable dashboard link instead of a console log that disappears when the cell finishes
- Know where this repo's own `BaseAgent`/`SimpleAgent` classes fit in, so you can reuse them instead of rebuilding this boilerplate yourself later

## Prerequisites

- [Step 01 — Test Your Setup & First Agent](step_01_test_setup_and_first_agent.ipynb) completed, or at least a working `.env` with a chat model API key (`GEMINI_API_KEY` or `OPENAI_API_KEY`)
- Comfortable with basic Python classes (`__init__`, methods, `self`) — used briefly near the end when subclassing `BaseAgent`; skim [Step 00](step_00_setup_and_python_basics.ipynb) first if classes are new to you
- No specific topic needed yet — this notebook uses its own running examples, not your team's topic, since it's about the framework itself rather than your project

## Background

CrewAI is a framework for orchestrating role-playing, autonomous AI agents that collaborate on shared tasks. It has no dependency on LangChain — it talks to any LLM provider directly through `litellm` (the same mechanism behind this repo's `MODEL` env var).

CrewAI ships two building blocks: **Crews** (teams of agents, optimized for flexibility and delegation) and **Flows** (event-driven, step-by-step control, closer to what a framework like LangGraph's graphs give you). **This course scopes itself to Crews only** — see the [CrewAI Flows docs](https://docs.crewai.com/en/concepts/flows) if you want to go further on your own.

## How this works

This notebook is one continuous walkthrough, not a single exercise cell — each section below builds on the previous one. Run the cells in order.

### A plain LLM call has no memory

Same idea as Step 01's Part 3 and Step 03 — a plain `crewai.LLM` call, no agent involved yet. Each call is an independent request/response round trip:

In [1]:
import os

from dotenv import load_dotenv
from crewai import LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

print(llm.call(messages=[{"role": "user", "content": "Hi, I am Tim!"}]))
print(llm.call(messages=[{"role": "user", "content": "What was my name again?"}]))

Hi Tim! It’s great to meet you. How are you doing today? Is there anything I can help you with?


I don’t actually know your name, as I don’t have access to your personal identity, private files, or a memory of our past conversations outside of this current chat session.

If you’d like to tell me, I’ll be happy to remember it for the duration of our conversation!


### CrewAI's core abstractions

CrewAI structures agentic workflows around four building blocks — the same four labeled consistently across every notebook in this course from Step 09 onward:

- **`Agent`** — a `role`, `goal`, `backstory`, and (optionally) `tools` — CrewAI's version of a system prompt/persona
- **`Task`** — a `description` of the work assigned to an agent, and an `expected_output` describing what a good result looks like
- **`Crew`** — the collection of agents + tasks + a `process` for running them
- **`Process`** — the orchestration strategy: `sequential` (a fixed pipeline, used throughout this course) or `hierarchical` (a manager agent delegates dynamically — see the [Process docs](https://docs.crewai.com/en/concepts/processes) if you need it)

```
┌─────────┐
│  Start  │
└────┬────┘
     │
     ▼
┌──────────────┐
│ Agent + Task │
└──────┬───────┘
       │
       ▼
┌─────────┐
│   End   │
└─────────┘
```

When a `Crew` has multiple `Task`s, `Task(context=[other_task])` forwards a prior task's output into the next one — [Step 13](step_13_multi_agent_seq.ipynb) uses exactly this for handing work between agents.

In [2]:
from crewai import Agent, Task, Crew, Process

# ── Agent ─────────────────────────────────────────────────────────────────────
agent = Agent(
    role="Assistant",
    goal="Have a natural, helpful conversation with the user",
    backstory="You are a friendly, helpful assistant.",
    llm=llm,
)

# ── Task ──────────────────────────────────────────────────────────────────────
task = Task(
    description="Hi, I am Tim",
    expected_output="A natural, conversational reply to the user's message.",
    agent=agent,
)

# ── Crew ──────────────────────────────────────────────────────────────────────
crew = Crew(agents=[agent], tasks=[task], process=Process.sequential)

# ── Process — kick off the crew ────────────────────────────────────────────────
result = crew.kickoff()
print(result.raw)

Hi Tim! It's nice to meet you. How is your day going so far? Is there anything I can help you with today?


### Seeing what happened: tracing

`verbose=True` prints a live log while a `Crew` runs, but it's gone once the cell finishes — nothing to revisit or share afterward. Passing `tracing=True` to a `Crew` instead uploads that same information — agent reasoning, task timing, tool calls — to a hosted dashboard and prints a link to it, `https://app.crewai.com/...`. No signup is needed to view it; the access code baked into the URL is enough. It's the more useful way to actually look at what a `Crew` did, especially once a run involves more than one agent or task.

This notebook doesn't demonstrate it live, on purpose: in the installed version of CrewAI, the plain `llm.call()` at the very top of this notebook — running before any `Crew` exists — leaves tracing unable to print a link for a later `Crew(tracing=True)`, even with nothing wrong in the code itself. [Step 13](step_13_multi_agent_seq.ipynb) doesn't hit this, since it starts straight with a `Crew`, and uses `tracing=True` on its `Crew` specifically so you have a trace timeline to inspect. (Curious anyway? Restart this notebook's kernel and re-run only the cells from "CrewAI's core abstractions" onward, skipping the plain `llm.call()` demo above — then add `tracing=True` to the `Crew(...)` call yourself.)

The upload happens on a background thread; a plain script naturally waits for it to finish at exit, but a notebook's kernel keeps running regardless — that's why Step 13 also calls `crewai_event_bus.flush()` right after `kickoff()`, to block until the upload is done so the link is guaranteed to print before the cell finishes.

This is the lightweight, no-account mode. If you want traces to persist across runs instead of being a one-off shareable link, `crewai login` plus a free CrewAI AMP account keeps a permanent history — or set `CREWAI_TRACING_ENABLED=true` in `.env` to turn tracing on for every `Crew` without passing the flag each time. Both are beyond what this course needs, but there if you want them.

### Giving it memory

Same problem as the plain LLM call: every `crew.kickoff()` above starts fresh, nothing carries over to the next call. Passing `memory=True` to a `Crew` fixes that — CrewAI stores recent interactions and retrieves them by semantic similarity on the next call. This needs an embedder (a model that turns text into vectors for that similarity search); point it at Gemini the same way the rest of this repo does (see `.env.example` and `src/research_crew/crew.py`). One gotcha worth knowing: the Gemini embedder's model setting shares a "model" alias with this repo's own `MODEL` env var, so it silently inherits `MODEL`'s value unless pinned explicitly via `EMBEDDINGS_GOOGLE_GENERATIVE_AI_MODEL_NAME`:

In [3]:
os.environ.setdefault("EMBEDDINGS_GOOGLE_GENERATIVE_AI_MODEL_NAME", "gemini-embedding-001")

embedder = {
    "provider": "google-generativeai",
    "config": {"api_key": os.getenv("GEMINI_API_KEY")},
}


def ask(message: str) -> str:
    task = Task(
        description=message,
        expected_output="A natural, conversational reply to the user's message.",
        agent=agent,
    )
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, memory=True, embedder=embedder)
    return crew.kickoff().raw


print(ask("Hi, my name is Tim and I love programming in Python!"))
print(ask("What's my name and what do I love?"))

/Users/jgehbauer/Coding/research_crew/.venv/lib/python3.12/site-packages/chromadb/utils/embedding_functions/google_embedding_function.py:145: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Hi Tim! It's great to meet you. Python is such a fantastic language—it's so versatile and readable, which makes it a joy to work with whether you're building web apps, diving into data science, or just automating little tasks.

How long have you been programming with it? Are you currently heads-down on an interesting project, or do you have a favorite library or framework that you really enjoy using?


Your name is Tim, and you have a real passion for programming in Python! It’s a great hobby to have—has that passion led you to work on any fun projects lately?


#### Good to know

- Memory is scoped **per agent `role`**, not per Python object or "conversation" — two `Crew`s reusing the same `Agent` share memory; two `Agent`s with different `role`s never do
- `crew.reset_memories("all")` clears it
- Recall is semantic/approximate (via the embedder), not an exact replay of past messages — don't rely on it for anything that needs to be verbatim-correct
- This is CrewAI's *short-term* memory. There's also long-term (SQLite, persists across restarts), entity (facts about named things), and external (bring-your-own backend, e.g. Mem0) — see the [CrewAI Memory docs](https://docs.crewai.com/en/concepts/memory) for all of it, including production/storage considerations

### Reusing this: `BaseAgent` and `SimpleAgent`

Wrapping this `Agent`/`Task`/`Crew` boilerplate in a class is common enough that this repo ships one at [src/research_crew/agents/base.py](../../src/research_crew/agents/base.py), plus a ready-to-use `SimpleAgent` on top of it — including the memory setup above. Subclass `BaseAgent` when you want your own persona:

In [4]:
from research_crew.agents import BaseAgent


class Pirate(BaseAgent):
    def __init__(self, llm=None, **kwargs):
        self.role = "Pirate Assistant"
        self.goal = "Have a helpful, natural conversation with the user"
        self.backstory = "You are a helpful assistant who talks like a pirate."
        super().__init__(llm=llm, **kwargs)


pirate = Pirate(llm=llm, memory=True, embedder=embedder)
print(pirate.run("Hi, I am Tim!").raw)
print(pirate.run("What was my name again?").raw)

Ahoy there, Tim! 'Tis a pleasure to make yer acquaintance. Pull up a barrel and rest yer boots—what be on yer mind this fine day? I'm ready to lend a hand with whatever plunder or tasks ye be lookin' to tackle!


Shiver me timbers, Tim! Ye must have had a bit too much grog if ye’ve forgotten already—yer name be Tim, matey! Don't ye worry none, though; this old pirate’s memory is as sharp as a cutlass. Is there anything else I can help ye with on this journey?


## Your task

There's no team-topic exercise here — this notebook is about the framework itself. Instead:

1. Run every cell in order, from top to bottom, and read each output before moving to the next cell.
2. Write your own one-paragraph subclass of `BaseAgent` (like `Pirate` above) with a persona of your choosing, and have a short back-and-forth with it.

## Shortcomings

Everything above was assembled directly in Python, inline, in one notebook cell each time — fine for learning, but not how you'd want to maintain a real project. `BaseAgent`/`SimpleAgent` start to fix that; CrewAI's memory recall being semantic/approximate rather than an exact replay of past turns doesn't.

This repo also ships a fuller reference project at `src/research_crew/`, where `role`/`goal`/`backstory` and task definitions live in YAML config instead of Python — see the main [README](../../README.md#the-template-code) for how that's organized. [Step 09](step_09_single_agent.ipynb) picks the `Agent` thread back up on a real research topic, and [Step 13](step_13_multi_agent_seq.ipynb) is where a second agent and a real `Crew` enter the picture for good, once Steps 10–12 have added tools, MCP, and RAG to the single-agent line first.

## Resources for further reading

- [CrewAI documentation](https://docs.crewai.com) — the full concept reference (agents, tasks, processes, tools, memory, knowledge, flows)
- [CrewAI Flows docs](https://docs.crewai.com/en/concepts/flows) — the other half of CrewAI this course doesn't cover
- [CrewAI Memory docs](https://docs.crewai.com/en/concepts/memory) — the short-term/long-term/entity/external memory system introduced above
- [CrewAI Process docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs `hierarchical`, and the manager-agent pattern
- [CrewAI Tracing docs](https://docs.crewai.com/en/observability/tracing) — the `tracing=True` dashboard introduced above, including the persistent-account mode
- [CrewAI GitHub repository](https://github.com/crewAIInc/crewAI)

That's it for Step 02 — CrewAI's `Agent`/`Task`/`Crew`/`Process` are the exact same four abstractions every remaining notebook in this course uses, just with the topic and roles changed. On to [Step 03 — Zero-Shot Prompting](step_03_zero_shot_prompting.ipynb).